# Industrial Visual Anomaly Detection — Project Overview Notebook

**Goal:** one notebook that lets you review the entire project from **data quality → preprocessing → CNN features → memory bank → coreset → inference → metrics → debugging**.

This notebook is designed mainly as a **study / interview overview**.  
It uses the same terminology and logic as the scripts in your project.

> Important: some cells are computationally heavy. Keep `RUN_HEAVY_STEPS = False` when reviewing.  
> Turn it on only when the 3CAD dataset and model folders are available locally.

## End-to-end flow

```text
Raw 3CAD dataset
    ↓
Data quality audit
    ├─ corrupted/unreadable images
    ├─ resolution distribution
    ├─ exact duplicates
    └─ train-test leakage
    ↓
Preprocessing audit
    ├─ V1: Resize + CenterCrop224
    └─ V2/V2c: Letterbox256 + valid-region mask
    ↓
Pretrained ResNet18
    ↓
Intermediate CNN features
    ├─ layer2: more spatial/local detail
    └─ layer3: stronger context
    ↓
Feature fusion
    ↓
Feature map → local patch features
    ↓
L2 normalization
    ↓
Normal-only memory bank
    ↓
10% greedy coreset
    ↓
Test patch → nearest normal feature
    ↓
Nearest-normal distance = anomaly score
    ↓
Image score = max patch score
    ↓
Threshold → Good / Defect
    ↓
AUROC / AP / Precision / Recall / F1
    ↓
FP / FN → Overkill / Underkill
    ↓
Error analysis + iteration
```

## 0. Project versions and final recorded results

| Version | Main change | AUROC | AP | Precision | Recall | F1 | FP | FN |
|---|---|---:|---:|---:|---:|---:|---:|---:|
| V1 | CenterCrop224, 16×16-style multilayer features | 0.9210 | 0.9726 | 0.8922 | 0.9294 | 0.9104 | 121 | 76 |
| V2a | Letterbox256, 16×16 fusion | 0.8991 | 0.9659 | 0.9194 | 0.8793 | 0.8989 | 83 | 130 |
| V2b | 32×32 test grid but stride-2 training memory | 0.4876 | 0.7267 | 0.7453 | 1.0000* | 0.8541 | 368 | 0 |
| V2c | Letterbox256 + full 32×32 train/test coverage | **0.9166** | **0.9716** | 0.8836 | **0.9304** | **0.9064** | 132 | **75** |

`*` V2b recall=1.0 is misleading because almost every good image was predicted as defect.

### Important project findings

- CenterCrop caused **95 / 1,077** defect masks to become fully empty.
- Letterbox256 reduced fully-lost defects to **0**.
- Final V2c full memory bank: **744,000 × 384 ≈ 1.09 GB**.
- 10% greedy coreset: **74,400 × 384 ≈ 109 MB**.
- V2b failure was traced to **train-test feature coverage mismatch**.

# 1. Imports and configuration

The original project is split across many scripts.  
This notebook keeps the **important reusable pieces** in one place.

Original script groups:

- `src/00_data_audit/` — dataset quality and duplicate checks
- `src/01_feature_experiments/` — global / patch feature experiments
- `src/v1_centercrop224/` — V1 CenterCrop baseline
- `src/preprocessing_audit/` — crop-loss audit
- `src/v2_letterbox256/` — Letterbox and V2/V2c experiments

In [2]:
from pathlib import Path
from collections import Counter, defaultdict
import hashlib
import csv
import time

import numpy as np
import torch
import torch.nn.functional as F

from PIL import Image

from torchvision import models
from torchvision.transforms.functional import pil_to_tensor, normalize
from torchvision.transforms import Compose, Resize, CenterCrop, InterpolationMode

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_recall_curve,
    confusion_matrix,
)

# ------------------------------------------------------------
# Adjust this path to your local project
# ------------------------------------------------------------
PROJECT_ROOT = Path(r"C:\Users\User\Documents\Projects\industrial-defect-ai")

DATA_DIR = PROJECT_ROOT / "data" / "3CAD"
PRODUCT_NAME = "Aluminum_Camera_Cover"
PRODUCT_DIR = DATA_DIR / PRODUCT_NAME

TRAIN_DIR = PRODUCT_DIR / "train" / "good"
TEST_DIR = PRODUCT_DIR / "test"
GROUND_TRUTH_DIR = PRODUCT_DIR / "ground_truth"

MODEL_DIR = PROJECT_ROOT / "models"
REPORT_DIR = PROJECT_ROOT / "reports"

MODEL_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

TARGET_SIZE = 256
MIN_VALID_RATIO = 0.50

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Keep False for interview/study review.
RUN_HEAVY_STEPS = False

print("Device:", device)

Device: cuda


# 2. Data Quality Audit

Before modeling, check the dataset itself.

Why?

- corrupted images can break training/inference,
- inconsistent resolutions affect preprocessing,
- exact duplicates can inflate results,
- train-test duplicates create **data leakage**.

This corresponds mainly to:

- `1_inspect_data.py`
- `2_audit_images.py`
- `3_audit_duplicates.py`
- `4_audit_near_duplicates.py`

In [3]:
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}

def find_images(folder: Path, exclude_ground_truth=False):
    images = []
    for path in folder.rglob("*"):
        if not path.is_file():
            continue
        if path.suffix.lower() not in IMAGE_EXTENSIONS:
            continue
        if exclude_ground_truth and "ground_truth" in path.parts:
            continue
        images.append(path)
    return images


def audit_image_quality(folder: Path):
    images = find_images(folder)
    corrupted = []
    resolution_counts = Counter()
    mode_counts = Counter()

    for path in images:
        try:
            with Image.open(path) as img:
                img.verify()

            with Image.open(path) as img:
                resolution_counts[img.size] += 1
                mode_counts[img.mode] += 1

        except Exception:
            corrupted.append(path)

    print("Total image files:", len(images))
    print("Corrupted:", len(corrupted))
    print("Unique resolutions:", len(resolution_counts))
    print("Most common resolutions:", resolution_counts.most_common(10))
    print("Image modes:", mode_counts)

    return corrupted, resolution_counts


# Example:
# corrupted, resolutions = audit_image_quality(DATA_DIR)

## Exact duplicate / leakage audit

Two files with identical SHA-256 hashes have identical bytes.

The important distinction is:

- **duplicate**: repeated content anywhere,
- **train-test leakage**: the same image exists in both train and test.

In the project audit, exact train-test leakage was detected.  
For strict production-quality reporting, build a cleaned split and evaluate separately.

In [4]:
def calculate_sha256(file_path: Path) -> str:
    sha256 = hashlib.sha256()

    with open(file_path, "rb") as f:
        while True:
            chunk = f.read(1024 * 1024)
            if not chunk:
                break
            sha256.update(chunk)

    return sha256.hexdigest()


def get_split(path: Path) -> str:
    if "train" in path.parts:
        return "train"
    if "test" in path.parts:
        return "test"
    return "unknown"


def audit_exact_duplicates(data_dir: Path):
    images = find_images(data_dir, exclude_ground_truth=True)

    hash_to_files = defaultdict(list)

    for path in images:
        hash_to_files[calculate_sha256(path)].append(path)

    duplicate_groups = {
        h: paths
        for h, paths in hash_to_files.items()
        if len(paths) > 1
    }

    leakage_groups = []

    for paths in duplicate_groups.values():
        splits = {get_split(p) for p in paths}
        if "train" in splits and "test" in splits:
            leakage_groups.append(paths)

    print("Product images:", len(images))
    print("Exact duplicate groups:", len(duplicate_groups))
    print("Train-test leakage groups:", len(leakage_groups))

    return duplicate_groups, leakage_groups


# Example:
# duplicate_groups, leakage_groups = audit_exact_duplicates(DATA_DIR)

### Near-duplicate lesson

The project also tested pHash near-duplicate detection.

Important lesson: in industrial images, repeated global structure can make pHash report many suspicious pairs even when the actual defect differs.

So pHash should be treated as a **candidate generator**, not an automatic deletion rule.

# 3. Preprocessing Audit

## V1: ImageNet-style Resize + CenterCrop224

Problem found:

**95 / 1,077 defect masks became fully empty after CenterCrop.**

This is important because:

> a model cannot detect a defect that preprocessing has already removed.

For segmentation / defect masks, always use **nearest-neighbor interpolation**.

In [5]:
mask_transform_v1 = Compose([
    Resize(
        256,
        interpolation=InterpolationMode.NEAREST,
    ),
    CenterCrop(224),
])


def mask_fully_lost_after_centercrop(mask_path: Path):
    mask = Image.open(mask_path).convert("L")

    before = np.array(mask) > 127
    transformed = mask_transform_v1(mask)
    after = np.array(transformed) > 127

    return (
        before.sum() > 0
        and after.sum() == 0
    )

## V2 / V2c: Letterbox256

Letterbox:

1. preserve aspect ratio,
2. resize image to fit the target canvas,
3. pad the remaining region,
4. create a valid-region mask so padding-heavy CNN patches can be ignored.

The image uses bilinear resize.  
Ground-truth masks should use nearest-neighbor resize.

In [6]:
def letterbox_image(image: Image.Image, target_size=256):
    original_width, original_height = image.size

    scale = min(
        target_size / original_width,
        target_size / original_height,
    )

    new_width = max(1, int(round(original_width * scale)))
    new_height = max(1, int(round(original_height * scale)))

    resized = image.resize(
        (new_width, new_height),
        resample=Image.Resampling.BILINEAR,
    )

    # ImageNet mean in RGB uint8 space.
    mean_rgb = (0.485, 0.456, 0.406)
    padding_color = tuple(
        int(round(v * 255))
        for v in mean_rgb
    )

    canvas = Image.new(
        "RGB",
        (target_size, target_size),
        color=padding_color,
    )

    left = (target_size - new_width) // 2
    top = (target_size - new_height) // 2

    canvas.paste(resized, (left, top))

    # 1 = real image region
    # 0 = padding
    valid_region = torch.zeros(
        (1, target_size, target_size),
        dtype=torch.float32,
    )

    valid_region[
        :,
        top:top + new_height,
        left:left + new_width,
    ] = 1.0

    tensor = pil_to_tensor(canvas).float() / 255.0

    tensor = normalize(
        tensor,
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    )

    return tensor, valid_region

# 4. CNN Backbone — ResNet18

ResNet18 does **not** only have 4 layers.  
PyTorch groups its residual blocks into four large stages:

```text
conv1
↓
maxpool
↓
layer1
↓
layer2
↓
layer3
↓
layer4
↓
global average pooling
↓
fully connected layer
```

For 256×256 input:

- `layer2` ≈ **32×32×128**
- `layer3` ≈ **16×16×256**

Why layer2 + layer3?

- **layer2:** stronger spatial/local texture detail,
- **layer3:** stronger contextual representation,
- **layer4:** more abstract but spatially coarser, which may hurt tiny-defect sensitivity.

In [7]:
weights = models.ResNet18_Weights.DEFAULT

model = models.resnet18(
    weights=weights
).to(device)

model.eval()


def extract_highres_features(x):
    # Initial stem
    x = model.conv1(x)
    x = model.bn1(x)
    x = model.relu(x)
    x = model.maxpool(x)

    x = model.layer1(x)

    # 256 input -> layer2 ≈ 32×32×128
    layer2 = model.layer2(x)

    # layer3 ≈ 16×16×256
    layer3 = model.layer3(layer2)

    # V2c:
    # preserve layer2's 32×32 grid,
    # upsample layer3 to 32×32.
    layer3_up = F.interpolate(
        layer3,
        size=layer2.shape[-2:],
        mode="bilinear",
        align_corners=False,
    )

    # 128 + 256 = 384 dimensions
    fused = torch.cat(
        [layer2, layer3_up],
        dim=1,
    )

    return fused

# 5. Feature Map → Patch Features

A patch feature is **not literally a separate cropped image**.

It is the CNN descriptor at one spatial feature-map position.  
Each position represents a receptive field in the original image.

V2c fused output:

```text
[1, 384, 32, 32]
```

Flattened:

```text
1024 patches × 384 features
```

Then apply **L2 normalization**.

In [8]:
def feature_map_to_patches(feature_map):
    _, channels, h, w = feature_map.shape

    patches = (
        feature_map
        .permute(0, 2, 3, 1)
        .reshape(h * w, channels)
    )

    patches = F.normalize(
        patches,
        p=2,
        dim=1,
    )

    return patches, h, w


def valid_region_to_patch_mask(
    valid_region,
    h,
    w,
    min_valid_ratio=0.50,
):
    valid_region = (
        valid_region
        .unsqueeze(0)
        .to(device)
    )

    valid_ratio = (
        F.adaptive_avg_pool2d(
            valid_region,
            output_size=(h, w),
        )
        .reshape(-1)
    )

    return valid_ratio >= min_valid_ratio

### L2 normalization vs L2 regularization

Do not confuse them:

- **L2 normalization:** scales each feature vector to unit length.
- **L2 regularization / weight decay:** penalizes large model weights during training.

For normalized vectors:

```text
dot product = cosine similarity

||a - b||² = 2 - 2 cos(a,b)
```

# 6. Build the Normal Memory Bank

This is the core one-class idea:

- training uses **normal / good images only**,
- extract all valid normal patch features,
- store them as the normal reference distribution.

Final V2c:

```text
744,000 × 384 features
≈ 1.09 GB float32
```

This cell follows the logic of:

`31_build_full32_memory_bank.py`

In [9]:
def build_full32_memory_bank(train_dir: Path):
    image_paths = sorted(train_dir.glob("*.png"))
    all_features = []

    for i, image_path in enumerate(image_paths, start=1):
        image = Image.open(image_path).convert("RGB")

        tensor, valid_region = letterbox_image(
            image,
            target_size=TARGET_SIZE,
        )

        tensor = tensor.unsqueeze(0).to(device)

        with torch.no_grad():
            feature_map = extract_highres_features(tensor)

            patches, h, w = feature_map_to_patches(
                feature_map
            )

            valid_mask = valid_region_to_patch_mask(
                valid_region,
                h,
                w,
                min_valid_ratio=MIN_VALID_RATIO,
            )

            valid_patches = patches[valid_mask]

        all_features.append(
            valid_patches.cpu()
        )

        if i % 50 == 0:
            print(
                f"{i}/{len(image_paths)} images processed"
            )

    memory_bank = torch.cat(
        all_features,
        dim=0,
    )

    return memory_bank


if RUN_HEAVY_STEPS:
    full_memory_bank = build_full32_memory_bank(TRAIN_DIR)
    print("Memory bank:", full_memory_bank.shape)

# 7. Greedy 10% Coreset

Why?

The full memory bank is expensive for nearest-neighbor inference.

V2c:

```text
744,000 × 384
↓ traditional greedy 10%
74,400 × 384
```

Memory:

```text
~1.09 GB → ~109 MB
```

The project uses:

1. L2-normalized memory features,
2. random projection `384 → 64`,
3. greedy farthest-first selection.

This is a computationally heavy step.

In [10]:
def build_greedy_coreset(
    memory_bank,
    ratio=0.10,
    projection_dim=64,
    seed=42,
):
    memory_bank = F.normalize(
        memory_bank.float(),
        p=2,
        dim=1,
    )

    n, feature_dim = memory_bank.shape

    target_size = max(
        1,
        int(round(n * ratio)),
    )

    torch.manual_seed(seed)

    projection_matrix = (
        torch.randn(
            feature_dim,
            projection_dim,
            dtype=torch.float32,
        )
        / np.sqrt(projection_dim)
    )

    projected = memory_bank @ projection_matrix

    projected = F.normalize(
        projected,
        p=2,
        dim=1,
    )

    # Deterministic first representative
    selected_indices = [0]

    first = projected[0:1]

    # max_similarity[i] =
    # similarity to the best selected representative so far.
    max_similarity = (
        projected @ first.T
    ).squeeze(1)

    max_similarity[0] = 1.0

    for step in range(1, target_size):
        # Least-represented feature
        next_index = int(
            torch.argmin(max_similarity)
        )

        selected_indices.append(
            next_index
        )

        new_feature = projected[
            next_index:next_index + 1
        ]

        similarity_to_new = (
            projected @ new_feature.T
        ).squeeze(1)

        max_similarity = torch.maximum(
            max_similarity,
            similarity_to_new,
        )

        max_similarity[
            selected_indices
        ] = 1.0

        if (step + 1) % 1000 == 0:
            print(
                f"{step+1:,}/{target_size:,}"
            )

    selected_indices = torch.tensor(
        selected_indices,
        dtype=torch.long,
    )

    coreset = memory_bank[
        selected_indices
    ]

    return coreset, selected_indices


if RUN_HEAVY_STEPS:
    coreset, selected_idx = build_greedy_coreset(
        full_memory_bank,
        ratio=0.10,
    )
    print("Coreset:", coreset.shape)

# 8. Inference — Nearest Normal Feature

For each test patch:

1. extract normalized feature,
2. compare against the normal coreset,
3. find the **nearest / most similar normal feature**,
4. convert similarity to distance,
5. larger distance = more anomalous.

Image score in this project:

```text
image_score = maximum patch anomaly score
```

That means **one strongly anomalous patch can flag the whole image**.

In [11]:
def infer_image_score(
    image_path: Path,
    memory_bank: torch.Tensor,
):
    image = Image.open(
        image_path
    ).convert("RGB")

    tensor, valid_region = letterbox_image(
        image,
        target_size=TARGET_SIZE,
    )

    tensor = tensor.unsqueeze(0).to(device)

    memory_bank = F.normalize(
        memory_bank.float().to(device),
        p=2,
        dim=1,
    )

    with torch.no_grad():
        feature_map = extract_highres_features(
            tensor
        )

        patches, h, w = feature_map_to_patches(
            feature_map
        )

        valid_mask = valid_region_to_patch_mask(
            valid_region,
            h,
            w,
            min_valid_ratio=MIN_VALID_RATIO,
        )

        valid_patches = patches[
            valid_mask
        ]

        # Cosine similarity for normalized features
        similarity = (
            valid_patches
            @ memory_bank.T
        )

        nearest_similarity = (
            similarity
            .max(dim=1)
            .values
        )

        # Equivalent Euclidean distance
        distance_squared = (
            2.0
            - 2.0 * nearest_similarity
        )

        patch_scores = torch.sqrt(
            torch.clamp(
                distance_squared,
                min=0.0,
            )
        )

        image_score = (
            patch_scores
            .max()
            .item()
        )

    return image_score

# 9. Image-Level Evaluation

Important metrics:

- **Precision** = among predicted defects, how many are real defects?
- **Recall** = among real defects, how many did we detect?
- **F1** = balance between precision and recall.
- **AUROC** = ranking ability across thresholds.
- **Average Precision** = positive-focused ranking metric.

Industrial meaning:

- **FP = Overkill**  
  good unit predicted as defect → unnecessary rejection/review.

- **FN = Underkill / Escape**  
  defective unit predicted as good → defect may pass downstream.

For production, threshold should be calibrated on a **validation/calibration set**, not selected on the final test set.

In [12]:
def evaluate_scores(y_true, y_score):
    y_true = np.asarray(y_true)
    y_score = np.asarray(y_score)

    auroc = roc_auc_score(
        y_true,
        y_score,
    )

    ap = average_precision_score(
        y_true,
        y_score,
    )

    precision_values, recall_values, thresholds = (
        precision_recall_curve(
            y_true,
            y_score,
        )
    )

    p = precision_values[:-1]
    r = recall_values[:-1]

    f1_values = (
        2 * p * r
        / np.maximum(p + r, 1e-12)
    )

    best_i = int(
        np.argmax(f1_values)
    )

    best_threshold = float(
        thresholds[best_i]
    )

    y_pred = (
        y_score >= best_threshold
    ).astype(np.uint8)

    tn, fp, fn, tp = (
        confusion_matrix(
            y_true,
            y_pred,
            labels=[0, 1],
        )
        .ravel()
    )

    precision = (
        tp / (tp + fp)
        if tp + fp > 0
        else 0.0
    )

    recall = (
        tp / (tp + fn)
        if tp + fn > 0
        else 0.0
    )

    f1 = (
        2 * precision * recall
        / (precision + recall)
        if precision + recall > 0
        else 0.0
    )

    return {
        "AUROC": auroc,
        "AP": ap,
        "threshold": best_threshold,
        "precision": precision,
        "recall": recall,
        "F1": f1,
        "TN": int(tn),
        "FP": int(fp),
        "FN": int(fn),
        "TP": int(tp),
    }

## Threshold trade-off

Lower threshold usually means:

```text
more samples predicted as defect
→ Recall ↑
→ FN ↓
→ FP ↑
→ Precision may ↓
```

Higher threshold usually means:

```text
fewer samples predicted as defect
→ FP ↓
→ Precision may ↑
→ FN ↑
→ Recall ↓
```

There is no universally best threshold.  
The operating point depends on the business cost of **underkill vs overkill**.

# 10. Pixel-Level Localization

Image-level question:

> Is this image defective?

Pixel-level question:

> Where is the defect?

These are different tasks and need separate thresholds.

Important pixel metrics:

```text
Pixel Precision = TP / (TP + FP)

Pixel Recall = TP / (TP + FN)

Dice / Pixel F1 = 2TP / (2TP + FP + FN)

IoU = TP / (TP + FP + FN)
```

V1 pixel-level result showed localization was much harder than image classification:

- Precision ≈ 0.3362
- Recall ≈ 0.5360
- Dice/F1 ≈ 0.4133
- IoU ≈ 0.2604

This helped identify **tiny-defect / spatial-resolution sensitivity** as a bottleneck.

# 11. Error Analysis

Do not stop at overall AUROC.

Inspect:

- FP images,
- FN images,
- miss rate by defect type,
- score distributions,
- defect size / location,
- preprocessing artifacts.

Example lesson from the project:

V2a:

```text
FP: 83
FN: 130
```

Compared with V1:

```text
FP: 121
FN: 76
```

So V2a reduced **overkill** but increased **underkill**.

That is why one metric alone is not enough.

In [13]:
def confusion_counts(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    tp = ((y_true == 1) & (y_pred == 1)).sum()
    tn = ((y_true == 0) & (y_pred == 0)).sum()
    fp = ((y_true == 0) & (y_pred == 1)).sum()
    fn = ((y_true == 1) & (y_pred == 0)).sum()

    return {
        "TP": int(tp),
        "TN": int(tn),
        "FP": int(fp),
        "FN": int(fn),
    }

# 12. V2b Failure → V2c Debugging Story

This is one of the strongest interview stories from the project.

## V2b

Feature map:

```text
32×32
```

But training memory bank used:

```text
stride = 2
→ only part of spatial positions
```

Inference used:

```text
all 32×32 positions
```

Result:

```text
GOOD mean score   ≈ 0.8462
DEFECT mean score ≈ 0.8440
AUROC             ≈ 0.4876
```

Normal and defect distributions almost overlapped.

## Root cause

**Train-test feature coverage mismatch.**

Many normal test patches had no sufficiently similar normal reference features.

## V2c fix

Training and test both use:

```text
full 32×32 feature coverage
```

Result:

```text
GOOD mean score   ≈ 0.5633
DEFECT mean score ≈ 0.7269
AUROC             = 0.9166
Recall            = 0.9304
F1                = 0.9064
FN                = 75
```

### Interview lesson

```text
metric collapse
→ inspect score distributions
→ inspect pipeline consistency
→ form hypothesis
→ controlled fix
→ rerun evaluation
```

Do not immediately tune the threshold when the representation itself is broken.

# 13. 60-Second Interview Explanation

You can explain the project like this:

> I built an industrial visual anomaly detection system using the 3CAD dataset. I first performed data-quality checks, including corrupted-image checks, resolution analysis, duplicate detection, and train-test leakage analysis. I then audited the preprocessing and found that CenterCrop completely removed the defect region in 95 defect samples, so I changed the pipeline to Letterbox256 to preserve the full field of view.  
>
> For feature extraction, I used a pretrained ResNet18 and fused intermediate layer2 and layer3 features to balance local spatial detail and contextual information. I converted the feature map into normalized patch features and built a normal memory bank using only good training samples. I compressed the memory bank using a 10% greedy coreset.  
>
> During inference, every test patch is compared with its nearest normal feature, and the distance becomes the anomaly score. I use the maximum patch score as the image-level score and evaluate it using AUROC, Average Precision, Precision, Recall, F1, FP and FN.  
>
> My final V2c version achieved 0.9166 AUROC and 0.9304 recall. The project also taught me how important preprocessing, data leakage, overkill/underkill trade-offs, and train-test pipeline consistency are in industrial inspection.

# 14. Production / Real-Product Checklist

Before calling this production-ready:

- create a **clean leakage-free split**,
- create a separate **validation/calibration set**,
- choose threshold on validation, not test,
- evaluate per-product and per-defect-type recall,
- benchmark latency and GPU memory,
- accelerate nearest-neighbor search if needed,
- version:
  - preprocessing,
  - backbone,
  - memory bank / coreset,
  - threshold,
  - dataset,
- log:
  - anomaly-score distributions,
  - FP / FN,
  - latency,
  - model version,
- monitor data drift / camera / lighting changes.

The final project should be described as a **PatchCore-style / memory-bank-based anomaly detection system**, not an official reproduction of canonical PatchCore.

# 15. Original Script Map

Use this notebook for review; use the original scripts for full experiments.

| Stage | Original scripts |
|---|---|
| Dataset structure / quality | `1_inspect_data.py`, `2_audit_images.py` |
| Exact duplicates / leakage | `3_audit_duplicates.py` |
| Near duplicates | `4_audit_near_duplicates.py` |
| Global / patch similarity experiments | `5_test_feature_similarity.py`, `6_test_patch_similarity.py`, `7_visualize_patch_anomaly.py` |
| V1 memory / coreset / inference | `8`–`13` |
| V1 image evaluation / threshold / errors | `14`–`17` |
| V1 pixel threshold | `19_analyze_pixel_thresholds_fast.py` |
| CenterCrop loss audit | `20_audit_crop_loss.py` |
| Letterbox retention | `21_audit_letterbox_retention.py` |
| V2a Letterbox memory / coreset / evaluation | `22`–`27` |
| High-resolution diagnostic V2b | `28`–`30` |
| Final Full32 V2c | `31_build_full32_memory_bank.py`, `32_build_full32_coreset.py`, `33_evaluate_full32_image_level.py` |